# Challenge: Prostate Cancer Prediction ANN


In [3]:

import torch
import torch.nn as nn
import torch.optim as optim
import copy

# Set random seed for reproducibility
torch.manual_seed(42)

# # Input features: Age, PSA, MRI
# x = torch.tensor([[62.0, 2.9, 3.0]], dtype=torch.float32)

# Input features: Age, PSA, MRI (mocked dataset)
x = torch.tensor([
    [72, 3.8, 4],
    [69, 3.4, 4],
    [62, 2.9, 3],
    [67, 2.8, 3],
    [64, 2.7, 3],
    [63, 2.1, 2],
    [61, 1.6, 2],

    [55, 2.5, 2],
    [53, 2.0, 2],
    [50, 1.7, 1],
    [48, 1.4, 1],
    [46, 1.2, 1],
    [45, 0.9, 1],
    [43, 0.8, 1]
], dtype=torch.float32)

# Standardization
x_mean = x.mean(dim=0, keepdim=True)
x_std = x.std(dim=0, keepdim=True)
x = (x - x_mean) / x_std

# Labels
# [1, 0] = Prostate Cancer
# [0, 1] = Healthy
y = torch.tensor([
    [1.0, 0.0],
    [1.0, 0.0],
    [1.0, 0.0],
    [1.0, 0.0],
    [1.0, 0.0],
    [1.0, 0.0],
    [1.0, 0.0],

    [0.0, 1.0],
    [0.0, 1.0],
    [0.0, 1.0],
    [0.0, 1.0],
    [0.0, 1.0],
    [0.0, 1.0],
    [0.0, 1.0]
], dtype=torch.float32)

# Simple ANN model
# Architecture: 3 inputs -> 4 hidden neurons -> 2 outputs
model = nn.Sequential(
    nn.Linear(3, 4),
    nn.Sigmoid(),
    nn.Linear(4, 2),
    nn.Sigmoid()
)

# Loss function (Mean Squared Error)
criterion = nn.MSELoss()

# Optimizer using Stochastic Gradient Descent
optimizer = optim.SGD(model.parameters(), lr=0.05)

# Early stopping parameters
max_epochs = 10000      # Maximum number of training iterations
patience = 500          # Stop if no improvement for this many epochs
min_delta = 1e-6        # Minimum loss improvement required to reset patience

best_loss = float("inf")
best_epoch = 0
no_improve_count = 0

# Store the best model parameters
best_model_state = copy.deepcopy(model.state_dict())

print("Predicted output before training:")
with torch.no_grad():
    y_hat_before = model(x)
    loss_before = criterion(y_hat_before, y)
    print(y_hat_before)
    print("Loss before training:", loss_before.item())

print("\nStart training...\n")

# Training loop
for epoch in range(1, max_epochs + 1):

    # Forward pass: compute prediction
    y_hat = model(x)

    # Compute loss
    loss = criterion(y_hat, y)

    # Clear previous gradients
    optimizer.zero_grad()

    # Backpropagation: compute gradients
    loss.backward()

    # Update model parameters using gradient descent
    optimizer.step()

    current_loss = loss.item()

    # Check if the current loss improved significantly
    if best_loss - current_loss > min_delta:
        best_loss = current_loss
        best_epoch = epoch
        no_improve_count = 0

        # Save best model parameters
        best_model_state = copy.deepcopy(model.state_dict())
    else:
        no_improve_count += 1

    if epoch % 1000 == 0 or epoch == 1:
        print(f"Epoch {epoch:5d} | Loss = {current_loss:.10f}")

    if no_improve_count >= patience:
        print(f"\nEarly stopping triggered at epoch {epoch}")
        print(f"Best loss = {best_loss:.10f} at epoch {best_epoch}")
        break

# Restore best model parameters
model.load_state_dict(best_model_state)

print("\nPredicted output after restoring best model:")
with torch.no_grad():
    y_hat_after = model(x)
    loss_after = criterion(y_hat_after, y)
    print(y_hat_after)
    print("Best restored loss:", loss_after.item())

print("\nFinal learned parameters (best model):")
torch.set_printoptions(threshold=float("inf"))
for name, param in model.named_parameters():
    print(f"\n{name}")
    print("Shape:", param.shape)
    print(param.data)


Predicted output before training:
tensor([[0.4692, 0.3932],
        [0.4640, 0.3951],
        [0.4588, 0.4040],
        [0.4635, 0.3976],
        [0.4593, 0.4011],
        [0.4604, 0.4043],
        [0.4528, 0.4071],
        [0.4537, 0.4148],
        [0.4457, 0.4175],
        [0.4466, 0.4240],
        [0.4406, 0.4269],
        [0.4358, 0.4296],
        [0.4315, 0.4309],
        [0.4280, 0.4334]])
Loss before training: 0.24372872710227966

Start training...

Epoch     1 | Loss = 0.2437287271
Epoch  1000 | Loss = 0.1110161394
Epoch  2000 | Loss = 0.0527735017
Epoch  3000 | Loss = 0.0317791365
Epoch  4000 | Loss = 0.0206887759
Epoch  5000 | Loss = 0.0143489642
Epoch  6000 | Loss = 0.0105324388
Epoch  7000 | Loss = 0.0081039192
Epoch  8000 | Loss = 0.0064746612
Epoch  9000 | Loss = 0.0053297332
Epoch 10000 | Loss = 0.0044929651

Predicted output after restoring best model:
tensor([[0.9810, 0.0178],
        [0.9799, 0.0188],
        [0.9388, 0.0601],
        [0.9745, 0.0246],
        [0.9636


# Output Interpretation

The neural network produces a predicted output before training and after the training process completes.

Before training, the model generates predictions based on randomly initialized weights. Because the parameters are initialized randomly, the predicted values may differ each time the program is executed.

During training, the model repeatedly performs forward propagation, computes the loss, calculates gradients through backpropagation, and updates the parameters using gradient descent.

Instead of performing only one update step, the training loop runs for multiple epochs until an early stopping condition is triggered. Early stopping monitors the loss value and stops training when the loss no longer improves significantly for a certain number of iterations (defined by the patience parameter).

As training progresses, the loss value gradually decreases, indicating that the model is learning parameter values that produce predictions closer to the true label.

After training finishes, the model restores the parameters from the epoch that achieved the lowest loss during training. The final predicted output is therefore produced by the best-performing version of the model.

Overall, this demonstrates how gradient descent iteratively improves the neural network’s predictions by minimizing the loss function.



# Talking Points

## 1. How do we compute the Loss Function?

The loss function measures the difference between the predicted value (ŷ) and the actual value (y).

In this example we use Mean Squared Error (MSE):

Loss = (y − ŷ)²

A smaller loss means the prediction is closer to the true value.

---

## 2. How do we find the value of y?

The value y represents the true label from the training dataset.

For the prostate cancer example:

[1, 0] → Prostate Cancer  
[0, 1] → Healthy

These labels represent the correct outcome that the neural network should learn.

---

## 3. How do we compare y with ŷ?

After the forward pass, the model produces a predicted value called ŷ (y_hat).

We compare the predicted value with the true label using the loss function:

loss = criterion(y_hat, y)

This calculation quantifies how far the prediction is from the correct label.

---

## 4. What is Gradient Descent?

Gradient Descent is an optimization algorithm used to minimize the loss function.

It updates the model parameters in the direction that reduces prediction error.

weight = weight − learning_rate × gradient

By repeatedly updating parameters, the neural network gradually improves its predictions.

---

## 5. How do we use Gradient Descent with Backpropagation?

Backpropagation computes the gradient of the loss with respect to each weight in the network.

Training steps:

1. Forward pass to compute predictions  
2. Compute loss between y and ŷ  
3. Backpropagation calculates gradients  
4. Gradient descent updates parameters

In code:

loss.backward()  
optimizer.step()

---

## 6. What is a Perceptron?

A perceptron is the basic computational unit of a neural network.

z = w₁x₁ + w₂x₂ + ... + wₙxₙ + b

The result is then passed through an activation function to produce the neuron output.

Artificial Neural Networks (ANNs) combine many perceptrons across layers to learn complex patterns in data.

---

## 7. Why do we use Early Stopping?

Early stopping monitors the training loss and stops the training process when the model stops improving.

If the loss does not improve for a certain number of epochs (patience), training stops automatically.

This helps:

• avoid unnecessary computation  
• prevent overfitting in larger datasets  
• keep the best-performing model parameters
